## **Data Processing With No Limits**

During hyperparameter tuning, iteration limits were imposed on SVM (10,000) and Neural Network (200) to prevent excessively long runtimes. However, these models may not have fully converged within those limits. Including many hyperparameter variations caused very long runtimes, so this notebook retests the best-found configuration only with a greatly increased iteration budget.

This notebook retests SVM and Neural Network using their best-found hyperparameter configurations but with greatly increased iteration limits (unlimited for SVM; 10,000 for NN). The goal is to give the under-converged models a fair chance and determine whether additional iterations improve the outcomes.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

import plotly.graph_objects as go
import plotly.express as px

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, accuracy_score, classification_report

from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier


In [2]:
%store -r
print("Variables restored successfully.")

Variables restored successfully.


In [3]:
print("Current Variables")
print(f"target                      : {target.shape}")
print(f"current_profession_encoded  : {current_profession_encoded.shape}")
print(f"age_group                   : {age_group.shape}")
print(f"education_level             : {education_level.shape}")
print(f"employment_status           : {employment_status.shape}")
print(f"dev_type_encoded            : {dev_type_encoded.shape}")
print(f"work_years (non-null)       : {work_years.notna().sum()}")
print(f"learn_years (non-null)      : {learn_years.notna().sum()}")
print(f"org_size_ordinal            : {org_size_ordinal.shape}")
print(f"work_tool_count             : {work_tool_count.shape}")
print(f"personal_tool_count         : {personal_tool_count.shape}")
print(f"geographic_regions_encoded  : {geographic_regions_encoded.shape}")
print(f"language_features           : {language_features.shape}")
print(f"database_features           : {database_features.shape}")
print(f"platform_features           : {platform_features.shape}")
print(f"webframe_features           : {webframe_features.shape}")
print(f"devenv_features             : {devenv_features.shape}")
print(f"collab_features             : {collab_features.shape}")
print(f"aimodel_features            : {aimodel_features.shape}")
print(f"ai_industry_use             : {ai_industry_use.shape}")
print(f"ai_learn_how                : {ai_learn_how.shape}")
print(f"learncodeai_encoded         : {learncodeai_encoded.shape}")
print(f"aiselect_encoded            : {aiselect_encoded.shape}")
print(f"aiagents_encoded            : {aiagents_encoded.shape}")
print(f"aiagentchange_encoded       : {aiagentchange_encoded.shape}")
print(f"ai_technical_use            : {ai_technical_use.shape}")
print(f"ai_knowledge                : {ai_knowledge.shape}")
print(f"ai_orchestration            : {ai_orchestration.shape}")
print(f"ai_observe_secure           : {ai_observe_secure.shape}")
print(f"ai_external                 : {ai_external.shape}")

Current Variables
target                      : (49191,)
current_profession_encoded  : (49191, 4)
age_group                   : (49191, 6)
education_level             : (49191, 8)
employment_status           : (49191, 5)
dev_type_encoded            : (49191, 21)
work_years (non-null)       : 42893
learn_years (non-null)      : 43042
org_size_ordinal            : (49191,)
work_tool_count             : (49191,)
personal_tool_count         : (49191,)
geographic_regions_encoded  : (49191, 19)
language_features           : (49191, 42)
database_features           : (49191, 30)
platform_features           : (49191, 42)
webframe_features           : (49191, 28)
devenv_features             : (49191, 27)
collab_features             : (49191, 25)
aimodel_features            : (49191, 17)
ai_industry_use             : (49191, 10)
ai_learn_how                : (49191, 13)
learncodeai_encoded         : (49191, 2)
aiselect_encoded            : (49191, 4)
aiagents_encoded            : (49191, 4)
aiage

## **All Features and Train/Test Split**

In [4]:
X = pd.concat([
    current_profession_encoded,
    age_group,
    education_level,
    employment_status,
    dev_type_encoded,
    geographic_regions_encoded,
    pd.DataFrame({'org_size': org_size_ordinal}),
    pd.DataFrame({'work_exp': work_years}),
    pd.DataFrame({'years_code': learn_years}),
    pd.DataFrame({'work_tools': work_tool_count}),
    pd.DataFrame({'personal_tools': personal_tool_count}),
    language_features,
    database_features,
    platform_features,
    webframe_features,
    devenv_features,
    collab_features,
    aimodel_features,
    ai_industry_use,
    ai_learn_how,
    learncodeai_encoded,
    aiselect_encoded,
    aiagents_encoded,
    aiagentchange_encoded,
    ai_technical_use,
    ai_knowledge,
    ai_orchestration,
    ai_observe_secure,
    ai_external,
], axis=1)

y = target

# Remove NaN and set median data
X_clean = X.copy()
X_clean = X_clean.fillna(0)

# 
for col in ['work_exp', 'years_code', 'work_tools', 'personal_tools', 'org_size']:
    X_clean[col] = X_clean[col].fillna(X_clean[col].median())

print(f"Full matrix: {X_clean.shape}")
print(f"Target: {y.shape}")
print(f"Target Ratio: {y.value_counts().to_dict()}")

Full matrix: (49191, 395)
Target: (49191,)
Target Ratio: {0: 34016, 1: 15175}


In [5]:
# Three-way split: 60% train, 20% validation, 20% test
# First split: separate test set (20%)
X_temp, X_test, y_temp, y_test = train_test_split(
    X_clean, y, test_size=0.2, random_state=42, stratify=y
)

# Second split: separate train and validation from remaining 80%
# 0.25 of 80% = 20% of total for validation, leaving 60% for training
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.25, random_state=42, stratify=y_temp
)

# Scale ONLY continuous features!!!
continuous_features = ['org_size', 'work_exp', 'years_code', 'work_tools', 'personal_tools']

scaler = StandardScaler()
X_train_scaled = X_train.copy()
X_val_scaled = X_val.copy()
X_test_scaled = X_test.copy()

# Fit scaler on training data only, transform all three sets
X_train_scaled[continuous_features] = scaler.fit_transform(X_train[continuous_features])
X_val_scaled[continuous_features] = scaler.transform(X_val[continuous_features])
X_test_scaled[continuous_features] = scaler.transform(X_test[continuous_features])

# Convert to numpy arrays
X_train_scaled = X_train_scaled.values
X_val_scaled = X_val_scaled.values
X_test_scaled = X_test_scaled.values

# Oversampling with SMOTE
# Performed Worse
# from imblearn.over_sampling import SMOTE
# smote = SMOTE(random_state=42)
# X_train_scaled, y_train = smote.fit_resample(X_train_scaled, y_train)

print(f"Train size: {X_train_scaled.shape}")
print(f"Validation size: {X_val_scaled.shape}")
print(f"Test size: {X_test_scaled.shape}")
print(f"\nTrain class balance: {pd.Series(y_train).value_counts().to_dict()}")
print(f"Validation class balance: {pd.Series(y_val).value_counts().to_dict()}")
print(f"Test class balance: {pd.Series(y_test).value_counts().to_dict()}")
print(f"\nScaled features: {continuous_features}")
print(f"One-hot encoded features remain as 0/1 (not scaled)")

Train size: (29514, 395)
Validation size: (9838, 395)
Test size: (9839, 395)

Train class balance: {0: 20409, 1: 9105}
Validation class balance: {0: 6803, 1: 3035}
Test class balance: {0: 6804, 1: 3035}

Scaled features: ['org_size', 'work_exp', 'years_code', 'work_tools', 'personal_tools']
One-hot encoded features remain as 0/1 (not scaled)


## **SVM with no Iteration Limit**

In [9]:
SVM_C = 1
kernel = 'rbf'

train_errors, val_errors = [], []
print(f"Testing SVM with kernel: {kernel}")

svm = SVC(kernel=kernel, C=SVM_C, random_state=42)
svm.fit(X_train_scaled, y_train)
train_errors.append(1 - svm.score(X_train_scaled, y_train))
val_errors.append(1 - svm.score(X_val_scaled, y_val))
print(f"  C=1  Train Error: {train_errors[-1]:.4f}  Validation Error: {val_errors[-1]:.4f}")

svm_predictions = svm.predict(X_test_scaled)
cm_svm = confusion_matrix(y_test, svm_predictions)
acc_svm = accuracy_score(y_test, svm_predictions)
report_svm = classification_report(y_test, svm_predictions, target_names=['Non-Remote', 'Remote'], output_dict=True)
tn_svm, fp_svm, fn_svm, tp_svm = cm_svm.ravel()

fig = go.Figure(data=go.Table(
    header=dict(values=['Metric', 'Non-Remote', 'Remote']),
    cells=dict(values=[
        ['Accuracy', 'Precision', 'Recall', 'F1 Score'],
        [f"{acc_svm:.4f}", f"{report_svm['Non-Remote']['precision']:.4f}", f"{report_svm['Non-Remote']['recall']:.4f}", f"{report_svm['Non-Remote']['f1-score']:.4f}"],
        [f"{acc_svm:.4f}", f"{report_svm['Remote']['precision']:.4f}", f"{report_svm['Remote']['recall']:.4f}", f"{report_svm['Remote']['f1-score']:.4f}"]
    ])
))
fig.update_layout(
    title=f'SVM (Kernel={kernel}, C={SVM_C}) - Classification Report',
    template='plotly_white',
    height=450
)
fig.show()

fig = go.Figure(data=go.Heatmap(
    z=cm_svm,
    x=['Non-Remote', 'Remote'],
    y=['Non-Remote', 'Remote'],
    colorscale='Blues',
    text=cm_svm,
    texttemplate="%{text}",
    hoverongaps=False
))
fig.update_layout(
    title='SVM Confusion Matrix',
    xaxis_title='Predicted Label',
    yaxis_title='True Label',
    template='plotly_white'
)
fig.show()

Testing SVM with kernel: rbf
  C=1  Train Error: 0.1559  Validation Error: 0.2611
  C=1  Train Error: 0.1559  Validation Error: 0.2611


## **Neural Network with No Iteration Limit**

In [ ]:
architecture = (256, 128, 64)
activation = 'relu'
alpha = 0.001

mlp_best = MLPClassifier(
    hidden_layer_sizes=architecture,
    activation=activation,
    solver='adam',
    alpha=alpha,
    max_iter=10000,
    random_state=42
)

mlp_best.fit(X_train_scaled, y_train)
mlp_predictions = mlp_best.predict(X_test_scaled)

cm_mlp = confusion_matrix(y_test, mlp_predictions)
acc_mlp = accuracy_score(y_test, mlp_predictions)
report_mlp = classification_report(y_test, mlp_predictions, target_names=['Non-Remote', 'Remote'], output_dict=True)

tn_mlp, fp_mlp, fn_mlp, tp_mlp = cm_mlp.ravel()

fig = go.Figure(data=go.Table(
    header=dict(values=['Metric', 'Non-Remote', 'Remote'], fill_color='paleturquoise', align='left'),
    cells=dict(values=[
        ['Accuracy', 'Precision', 'Recall', 'F1 Score'],
        [f"{acc_mlp:.4f}", f"{report_mlp['Non-Remote']['precision']:.4f}", f"{report_mlp['Non-Remote']['recall']:.4f}", f"{report_mlp['Non-Remote']['f1-score']:.4f}"],
        [f"{acc_mlp:.4f}", f"{report_mlp['Remote']['precision']:.4f}", f"{report_mlp['Remote']['recall']:.4f}", f"{report_mlp['Remote']['f1-score']:.4f}"],
    ], fill_color='lavender', align='left')
))
fig.update_layout(
    title=f'Neural Network - Classification Report',
    template='plotly_white',
    height=400
)
fig.show()

fig = go.Figure(data=go.Heatmap(
    z=cm_mlp,
    x=['Predicted Non-Remote', 'Predicted Remote'],
    y=['Actual Non-Remote',    'Actual Remote'],
    text=[[str(tn_mlp), str(fp_mlp)], [str(fn_mlp), str(tp_mlp)]],
    texttemplate='%{text}',
    colorscale='Blues',
    showscale=True
))
fig.update_layout(
    title=f'Neural Network - Confusion Matrix',
    template='plotly_white',
    height=450
)
fig.show()

## **Summary and Comparison**

The table below compares the test accuracy of each model under the original iteration-limited tuning setup versus the extended iteration budget used in this notebook.\

| Model | Iteration Limit (Tuning) | Iteration Limit (This Notebook) | Accuracy (Tuning) | Accuracy (Unlimited) |
|---|---|---|---|---|
| SVM | 10,000 | Unlimited (-1) | 0.7456 | 0.7429 |
| Neural Network | 200 | 10,000 | 0.7135 | 0.7135 |

The results show that the models performed slightly worse or identically this may  confirm that the original iteration limits were already sufficient for convergence and that the tuning notebook's reported accuracies are reliable.